# 07 — Circuit and hardware-resource analysis

The preceding numerical stages established the formulation hierarchy, quantified the reduction in logical QUBO size, characterized ideal-state QAOA and isolated finite-sampling effects. The present notebook does not introduce a device-noise model. Its purpose is instead to translate the QUBO interaction structure into circuit-level quantities that remain meaningful before choosing a specific processor.

For a diagonal QUBO Hamiltonian, every non-zero quadratic coefficient induces a pairwise (Z_iZ_j) interaction in the QAOA cost layer. The resulting interaction graph therefore determines both the number of two-qubit cost terms and the amount of parallelism available when disjoint interactions can be executed simultaneously. A second question is how well that logical graph aligns with representative sparse hardware-connectivity graphs. The latter is evaluated through shortest-path distances under a deterministic placement heuristic; these distances are reported as routing-pressure indicators and are not interpreted as compiled SWAP counts.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa.benchmarks import (
    FROZEN_QAOA_BENCHMARKS,
    build_frozen_benchmark,
)
from oec_qaoa.circuit_resources import (
    connectivity_metrics,
    cost_layer_metrics,
)

TOPOLOGIES = ["all_to_all", "grid", "ring", "line"]

## Logical cost-layer structure

Because all (Z_iZ_j) terms commute, the cost Hamiltonian does not require a sequential ordering at the logical level. Two interactions can, however, be executed in the same two-qubit time slot only when they do not share a qubit. The maximum degree (Delta) of the interaction graph is therefore a lower bound on the number of parallel slots required by one logical cost layer.

A constructive upper estimate is obtained by edge coloring the interaction graph. The implementation colors the corresponding line graph with a deterministic greedy strategy; the resulting number of colors is not claimed to be the exact edge chromatic number, but it provides a reproducible schedule whose depth can be compared directly with the lower bound.

In [ ]:
logical_rows = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, _, model = build_frozen_benchmark(benchmark)
    metrics = cost_layer_metrics(model)

    logical_rows.append(
        {
            "benchmark_id": benchmark.benchmark_id,
            "instance": instance.name,
            **metrics.to_dict(),
        }
    )

logical = pd.DataFrame(logical_rows)
logical

The quantities above describe one QAOA cost layer. At depth (p), the number of logical pairwise cost interactions scales as

[
p,N_{ZZ},
]

while the corresponding all-to-all two-qubit time-slot estimate scales between

[
p,Delta
]

and

[
p,D_{mathrm{greedy}},
]

before accounting for hardware routing or a particular native gate decomposition. This distinction is important because the number of QUBO couplings alone does not determine circuit depth: a dense interaction graph can also restrict the degree of parallel execution.

## Representative connectivity graphs

A logical interaction edge is directly executable only when the assigned physical qubits are adjacent in the hardware-connectivity graph. Rather than selecting a particular commercial processor, the analysis uses four generic connectivity classes: all-to-all, a two-dimensional rectangular grid, a ring and a line.

Logical qubits are assigned deterministically by matching high-degree logical vertices to central hardware vertices. For every required (Z_iZ_j) interaction, the shortest-path distance (d_{ij}) between the assigned physical qubits is then recorded. The principal quantities are the fraction of interactions already supported at distance one, the mean and maximum interaction distance, and

[
R_{mathrm{dist}}
=
sum_{(i,j)in E_{mathrm{QUBO}}}
max(d_{ij}-1,0).
]

(R_{mathrm{dist}}) is a routing-pressure proxy only. It is not a SWAP count, because an actual compiler can reuse permutations, parallelize routing operations and choose a different placement.

In [ ]:
rows = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, _, model = build_frozen_benchmark(benchmark)

    for topology in TOPOLOGIES:
        metrics = connectivity_metrics(model, topology)
        rows.append(
            {
                "benchmark_id": benchmark.benchmark_id,
                "instance": instance.name,
                **metrics.to_dict(),
            }
        )

connectivity = pd.DataFrame(rows)
connectivity

## Comparison across the frozen benchmarks

The hardware-resource interpretation is deliberately separated from solver performance. A benchmark with a larger optimal-state probability is not necessarily easier to embed, and an interaction graph that is easy to route does not imply that the variational optimizer will find a good state.

The analysis therefore treats logical cost complexity and variational performance as distinct dimensions. In the paper, the circuit-resource table can be read beside the ideal-QAOA and finite-sampling results without converting either quantity into a single aggregate score.

In [ ]:
summary = connectivity.merge(
    logical[
        [
            "benchmark_id",
            "logical_qubits",
            "zz_terms",
            "interaction_max_degree",
            "greedy_parallel_depth",
        ]
    ],
    on="benchmark_id",
    how="left",
)

summary[
    [
        "benchmark_id",
        "topology",
        "logical_qubits",
        "zz_terms",
        "interaction_max_degree",
        "greedy_parallel_depth",
        "directly_supported_fraction",
        "mean_interaction_distance",
        "max_interaction_distance",
        "distance_excess_sum",
    ]
]

## Scope of the hardware analysis

No noisy simulation is included. The quantities reported here are intended to answer a narrower and more reproducible question: once the OEC problem has been reduced to a path-QUBO, what logical pairwise structure must a gate-based implementation realize, and how strongly does that structure conflict with sparse connectivity?

A device-specific compilation study could refine these estimates by fixing a native gate set, calibration snapshot and routing algorithm, but such results would become processor- and date-dependent. For the present study, the formulation-level interaction graph, logical parallel-depth bounds and connectivity-distance proxies provide the hardware-facing information needed to complement the qubit and coupling counts already established in the resource-scaling analysis.